# ЗАДАЧА 1: Токенизация текстов

In [1]:
model_name = "distilbert-base-uncased"

In [2]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    model_name,
    output_attentions=True  # важно!
)
model.eval()

/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1102 10:21:41.538000 97395 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6594.20it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [5]:
#грузим токенизатор
from transformers import AutoModel
from transformers import AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

In [8]:
text = ['this is very important!']

In [9]:
result = tokenize_texts(text)

In [10]:
result

{'input_ids': tensor([[ 101, 2023, 2003, 2200, 2590,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

In [11]:
batch = tokenize_texts(["This movie was great!", "Terrible movie, waste of time."])
print(batch["input_ids"].shape)
print(batch["attention_mask"])

torch.Size([2, 9])
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


# ЗАДАЧА 2: Извлечение CLS-эмбеддингов

In [12]:
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11583.60it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
model.eval()

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [16]:
import numpy as np
import torch

def get_cls_embeddings(texts, batch_size=32):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]       # очередные batch_size текстов
        batch = tokenize_texts(batch_texts)                  # функция из задачи 1

        with torch.no_grad():                                # градиенты не нужны, только предсказание
            outputs = model(**batch)

        cls = outputs.last_hidden_state[:, 0, :]             # вектор [CLS] для каждого текста
        all_embeddings.append(cls.cpu().numpy())

    return np.vstack(all_embeddings)                         # склеиваем батчи в одну матрицу

In [17]:
texts = [
    "This movie was great!",
    "Terrible movie, waste of time.",
    "I love you!"
]

emb = get_cls_embeddings(texts, batch_size=2)
print(emb.shape)

(3, 768)


# ЗАДАЧА 3: Logistic Regression на эмбеддингах

In [18]:
from datasets import load_dataset
import pandas as pd

imdb = load_dataset("stanfordnlp/imdb")
df = imdb["train"].shuffle(seed=42).select(range(2000)).to_pandas()[["text", "label"]]

import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/imdb_2000.csv", index=False)

print(df.shape)
print(df["label"].value_counts())

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 748322.73 examples/s]


(2000, 2)
label
1    1000
0    1000
Name: count, dtype: int64


In [19]:
texts = df["text"].tolist()
y = df["label"].values

X = get_cls_embeddings(texts)
print(X.shape)   # (2000, 768)

(2000, 768)


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)   # (1600, 768) (400, 768)

(1600, 768) (400, 768)


In [23]:
   clf = LogisticRegression(max_iter=1000, n_jobs=-1)
   clf.fit(X_train, y_train)
   y_pred = clf.predict(X_test)

/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [24]:
print(classification_report(y_test, y_pred))

f1 = f1_score(y_test, y_pred, average="macro")
print(f"Macro F1: {f1:.4f}")

with open("baseline_results.txt", "w") as f:
    f.write(f"Baseline: DistilBERT CLS embeddings + LogisticRegression\n")
    f.write(f"Macro F1: {f1:.4f}\n")

              precision    recall  f1-score   support

           0       0.81      0.83      0.82       200
           1       0.83      0.81      0.82       200

    accuracy                           0.82       400
   macro avg       0.82      0.82      0.82       400
weighted avg       0.82      0.82      0.82       400

Macro F1: 0.8200
